<a href="https://colab.research.google.com/github/Ullas33/National-Air-Sea-Radar/blob/main/National_Air_%26_Sea_Radar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q streamlit streamlit-folium folium websocket-client requests pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 523.7/523.7 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 119.6 MB/s eta 0:00:00


In [20]:
%%writefile app.py
import streamlit as st
import folium
from streamlit_folium import st_folium
from folium.plugins import MarkerCluster
import requests
import websocket
import json
import time

# --- Page Configuration ---
st.set_page_config(page_title="National Air & Sea Radar", layout="wide")
st.title("🛰️ Real-Time Air & Sea Surveillance Dashboard")
st.markdown("Monitoring commercial airspace and maritime vessels around the Indian subcontinent.")

# --- Secure API Key Management ---
# The app tries to fetch keys securely from Streamlit Cloud Secrets.
# If it fails (e.g., local testing), it leaves them blank for manual UI input.
try:
    SECURE_AIRLABS_KEY = "2c9bd979-2bef-4e11-ad5f-419cee0b2f1d"
    SECURE_AIS_KEY = "1551fb9ee2ba9d4fffad6bf74a728eb37d61c6ac"
except (FileNotFoundError, KeyError):
    SECURE_AIRLABS_KEY = ""
    SECURE_AIS_KEY = ""

# --- Sidebar Controls ---
st.sidebar.header("Tracking Controls")
track_flights = st.sidebar.checkbox("Track Flights (AirLabs)", value=True)
track_ships = st.sidebar.checkbox("Track Ships (AIS)", value=False)

# Fallback UI inputs if secrets aren't set
if not SECURE_AIRLABS_KEY:
    SECURE_AIRLABS_KEY = st.sidebar.text_input("AirLabs API Key", type="password")
if not SECURE_AIS_KEY:
    SECURE_AIS_KEY = st.sidebar.text_input("AISStream API Key", type="password")

# --- Helper Functions ---
@st.cache_data(ttl=60)
def get_flight_data(api_key):
    # AirLabs bounding box for India roughly
    url = f"https://airlabs.co/api/v9/flights?api_key={api_key}&bbox=8.0,68.0,37.0,97.0"
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            return response.json().get('response', [])
        else:
            st.sidebar.error(f"AirLabs API Error: {response.status_code}")
    except Exception as e:
        st.sidebar.error(f"Flight API Error: {e}")
    return []

def get_ship_data(api_key, listen_time=5):
    vessels = {}
    bounding_boxes = [[[5.0, 65.0], [25.0, 95.0]]]
    subscribe_message = {
        "APIKey": api_key,
        "BoundingBoxes": bounding_boxes,
        "FilterMessageTypes": ["PositionReport"]
    }
    try:
        ws = websocket.create_connection("wss://stream.aisstream.io/v0/stream", timeout=3)
        ws.send(json.dumps(subscribe_message))
        start = time.time()
        while time.time() - start < listen_time:
            try:
                result = ws.recv()
                message = json.loads(result)
                if message["MessageType"] == "PositionReport":
                    mmsi = message["MetaData"]["MMSI"]
                    report = message["Message"]["PositionReport"]
                    vessels[mmsi] = report
            except websocket.WebSocketTimeoutException:
                continue
        ws.close()
    except Exception as e:
        st.sidebar.error(f"WebSocket Error: {e}")
    return vessels

# --- Map Generation ---
m = folium.Map(location=[22.0, 82.0], zoom_start=5, tiles='CartoDB dark_matter')
flight_cluster = MarkerCluster(name="Flights").add_to(m)
ship_cluster = MarkerCluster(name="Ships").add_to(m)

if track_flights:
    if not SECURE_AIRLABS_KEY:
        st.sidebar.warning("Please enter your AirLabs API key to track flights.")
    else:
        with st.spinner("Fetching live flight data..."):
            flights = get_flight_data(SECURE_AIRLABS_KEY)
            count = 0
            for f in flights:
                lat, lon = f.get('lat'), f.get('lng')
                callsign = f.get('flight_iata') or f.get('flight_icao') or "UNKNOWN"
                country = f.get('flag')

                if lat and lon:
                    color = "blue" if country == "IN" else "red"
                    folium.CircleMarker(
                        location=[lat, lon], radius=3, color=color, fill=True,
                        popup=f"Callsign: {callsign}<br>Country Code: {country}"
                    ).add_to(flight_cluster)
                    count += 1
            st.sidebar.success(f"Tracking {count} flights.")

if track_ships:
    if not SECURE_AIS_KEY:
        st.sidebar.warning("Please enter your aisstream.io API key to track ships.")
    else:
        with st.spinner("Listening to maritime WebSocket..."):
            ships = get_ship_data(SECURE_AIS_KEY)
            count = 0
            for mmsi, data in ships.items():
                lat, lon = data['Latitude'], data['Longitude']
                if lat <= 90 and lon <= 180:
                    folium.CircleMarker(
                        location=[lat, lon], radius=3, color="#00ffcc", fill=True,
                        popup=f"MMSI: {mmsi}<br>Speed: {data['Sog']} knots"
                    ).add_to(ship_cluster)
                    count += 1
            st.sidebar.success(f"Tracking {count} ships.")

st_folium(m, width=1000, height=600, returned_objects=[])
st.markdown("---")
st.caption("Data sources: AirLabs (Flights) | aisstream.io (Maritime). For portfolio demonstration purposes only.")

Overwriting app.py


In [21]:
import urllib
import subprocess

# Get the external IP of your Colab instance (needed for localtunnel password)
print("Password/Endpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

# Run Streamlit in the background and expose it via localtunnel
!streamlit run app.py & npx localtunnel --port 8501

Password/Endpoint IP for localtunnel is: 34.125.32.64
⠙⠹

⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧your url is: https://pretty-trams-yell.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.32.64:8501

  Stopping...
^C


In [5]:
print("Installing localtunnel...")
!npm install -g localtunnel
print("Localtunnel installed successfully.")

Installing localtunnel...
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧Localtunnel installed successfully.
